## Propósito
Filtrar **cada CSV de eventos** de esta fuente por las 16 alcaldías de CDMX y 2018 en adelante. No corregimos nulos ni eliminamos duplicados. Conservamos datos crudos intactos. Los catálogos se documentan aparte.

## Preparar lectura y catálogo geográfico

In [ ]:
from pathlib import Path
import zipfile, io, re, unicodedata
import pandas as pd
ARCHIVO_ORIGEN = Path("CARPETA_DATOS(3).zip") # Ajustar ruta a donde se descargó
SALIDA = Path("salidas")
SALIDA.mkdir(exist_ok=True)
PREFIJO = "CARPETA_DATOS/Kaggle/archive/datos/datos/conjunto_de_datos/conjunto_de_datos/"
FUENTE = "Kaggle"

def leer_csv(z, nombre):
    contenido = z.read(nombre)
    for enc in ("utf-8-sig", "latin1"):
        try:
            return pd.read_csv(io.BytesIO(contenido), encoding=enc, dtype=str, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer {nombre}")

def normalizar(s):
    s=unicodedata.normalize("NFKD",str(s).strip().upper())
    return " ".join("".join(c for c in s if not unicodedata.combining(c)).split())

ALCALDIAS = {normalizar(x) for x in ["Álvaro Obregón","Azcapotzalco","Benito Juárez","Coyoacán","Cuajimalpa de Morelos","Cuauhtémoc","Gustavo A. Madero","Iztacalco","Iztapalapa","La Magdalena Contreras","Miguel Hidalgo","Milpa Alta","Tláhuac","Tlalpan","Venustiano Carranza","Xochimilco"]}


## Filtrar y guardar cada CSV con inventario

In [ ]:
def mascara_cdmx_y_fecha(df):
    cols = {{str(c).strip().lower().lstrip("\ufeff"):c for c in df.columns}}
    if FUENTE in ("SSC", "C5"):
        if FUENTE == "SSC":
            col_alc=cols.get("alcaldia")
            col_fecha=cols.get("fecha_evento")
        else:
            col_alc=cols.get("alcaldia_catalogo") or cols.get("alcaldia_inicio")
            col_fecha=cols.get("fecha_creacion")
        if not col_alc or not col_fecha: raise ValueError("Faltan columnas de alcaldía/fecha: revisar archivo")
        geo=df[col_alc].fillna("").map(normalizar).isin(ALCALDIAS)
        fechas=pd.to_datetime(df[col_fecha],errors="coerce",format="mixed",dayfirst=True)
        tiempo=fechas.dt.year.ge(2018)
    elif FUENTE == "INEGI_ATUS":
        ent=cols.get("cve_ent")
        mun=cols.get("cve_mun")
        anio=cols.get("anio")
        if not all([ent,mun,anio]): raise ValueError("Faltan claves INEGI")
        geo=pd.to_numeric(df[ent],errors="coerce").eq(9) & pd.to_numeric(df[mun],errors="coerce").between(1,16)
        tiempo=pd.to_numeric(df[anio],errors="coerce").ge(2018)
    else:
        # IMPORTANTE: algunas tablas Kaggle tienen encabezados desplazados;
        # no filtrar hasta comprobar alineación de encabezados y valores.
        raise ValueError("Kaggle: revisar alineación de columnas antes de filtrar. No se genera un CSV incorrecto.")
    return geo.fillna(False),tiempo.fillna(False)

inventario=[]
with zipfile.ZipFile(ARCHIVO_ORIGEN) as z:
    nombres=[n for n in z.namelist() if n.startswith(PREFIJO) and n.lower().endswith('.csv')]
    for nombre in nombres:
        df=leer_csv(z,nombre)
        try:
            geo,tiempo=mascara_cdmx_y_fecha(df)
            limpio=df.loc[geo & tiempo].copy() # no se modifican valores ni se eliminan duplicados
            salida=SALIDA / f"{{FUENTE}}_{{Path(nombre).stem}}_ETAPA1.csv"
            limpio.to_csv(salida,index=False,encoding="utf-8-sig")
            estado="generado"
            nfinal=len(limpio)
        except ValueError as e:
            estado=f"pendiente_revision: {{e}}"
            nfinal=None
        inventario.append({{"archivo":nombre,"filas_originales":len(df),"filas_etapa1":nfinal,"estado":estado}})
reporte=pd.DataFrame(inventario)
display(reporte)
reporte.to_csv(SALIDA/f"{{FUENTE}}_inventario_etapa1.csv",index=False,encoding="utf-8-sig")
